# C2 — Les plus proches voisins

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| C — Reconnaître des chiffres manuscrits | 2 sur 4 | 2 h 30 | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le chiffre moyen résume chaque chiffre par une seule image, et perd la variété des écritures. Une idée opposée consiste à ne rien résumer du tout : pour lire une nouvelle image, on la compare à **toutes** les images d'apprentissage, et l'on regarde quel chiffre ont ses plus proches voisines.

**La sous-question de cette étape : la méthode des plus proches voisins fait-elle mieux ? Quelle mesure de ressemblance utiliser, combien de voisins consulter, et combien de temps le calcul prend-il ?**

**Activité** : Quels avantages et quels inconvénients prévoyez-vous pour cette méthode par rapport au chiffre moyen ? Pensez à la précision, à la mémoire nécessaire, au temps de calcul.

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_reussite_voisins = 0.95       # taux de réussite avec un seul voisin (modifiez)
pari_meilleure_mesure = "distance" # "distance" ou "cosinus" (modifiez)

## ③ Abstraire

**Les $k$ plus proches voisins.** Pour une image $\mathbf{x}$, on calcule sa distance à chacune des $N$ images d'apprentissage, on garde les $k$ plus proches, et l'on répond le chiffre le plus fréquent parmi elles (en cas d'égalité, le plus proche l'emporte).

**Distance ou cosinus ?** Deux mesures du module M1 sont disponibles. La distance $\lVert \mathbf{x} - \mathbf{y} \rVert$ compare tout, y compris l'intensité du trait. Le cosinus $\dfrac{\mathbf{x}\cdot\mathbf{y}}{\lVert \mathbf{x} \rVert\,\lVert \mathbf{y} \rVert}$ ne compare que la **direction** : un chiffre tracé plus pâlement, dont tous les pixels sont multipliés par $0{,}7$, a exactement le même cosinus avec les autres.

**Calculer vite.** Calculer $N$ distances en dimension 64 pour chaque image, avec des boucles, est lent. L'identité

$$
\lVert \mathbf{x} - \mathbf{y} \rVert^2 = \lVert \mathbf{x} \rVert^2 + \lVert \mathbf{y} \rVert^2 - 2\,\mathbf{x}\cdot\mathbf{y}
$$

ramène toutes les distances entre deux paquets d'images à un seul **produit de matrices** $X_{\text{test}}\,X_{\text{app}}^{\mathsf{T}}$, que NumPy calcule très efficacement.

**Activité** : Vérifiez l'identité à la main avec $\mathbf{x} = (1\,;\,2)$ et $\mathbf{y} = (3\,;\,1)$. Pourquoi faut-il ensuite remplacer les éventuels petits nombres négatifs par 0 avant de prendre la racine ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Mettre de côté un paquet de validation

Le paquet de test reste sous scellés jusqu'à l'étape C4. Pour choisir les réglages, on découpe le paquet d'apprentissage en deux.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from chiffres import charger, decouper, afficher, matrice_confusion

X, y = charger()
X_reste, y_reste, X_test, y_test = decouper(X, y, part_test=0.25, graine=0)       # test sous scellés
X_app, y_app, X_val, y_val = decouper(X_reste, y_reste, part_test=0.25, graine=1)
print(f"apprentissage {len(y_app)}, validation {len(y_val)}, test {len(y_test)} (sous scellés)")

### Deux façons de calculer les distances

In [ ]:
def distances_boucles(A, B):
    D = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            D[i, j] = np.sqrt(np.sum((A[i] - B[j]) ** 2))
    return D

def distances_matricielles(A, B):
    carres = (A ** 2).sum(axis=1)[:, None] + (B ** 2).sum(axis=1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.clip(carres, 0, None))

debut = time.perf_counter(); D1 = distances_boucles(X_val[:50], X_app); t1 = time.perf_counter() - debut
debut = time.perf_counter(); D2 = distances_matricielles(X_val[:50], X_app); t2 = time.perf_counter() - debut
print(f"boucles : {1000 * t1:.0f} ms ; produit de matrices : {1000 * t2:.1f} ms ; mêmes résultats ? {np.allclose(D1, D2)}")
print(f"Le produit de matrices est environ {t1 / t2:.0f} fois plus rapide.")

### Distance ou cosinus, et combien de voisins ?

In [ ]:
def similarites_cosinus(A, B):
    An = A / np.linalg.norm(A, axis=1, keepdims=True)
    Bn = B / np.linalg.norm(B, axis=1, keepdims=True)
    return An @ Bn.T

def voter(proches, y_app):
    # chiffre le plus fréquent parmi les voisins ; en cas d'égalité, celui du voisin le plus proche
    resultat = []
    for ligne in y_app[proches]:
        comptes = np.bincount(ligne, minlength=10)
        gagnants = np.flatnonzero(comptes == comptes.max())
        resultat.append(next(c for c in ligne if c in gagnants))
    return np.array(resultat)

def predire_voisins(X_nouveaux, X_app, y_app, k=1, mesure="distance"):
    if mesure == "distance":
        ordre = np.argsort(distances_matricielles(X_nouveaux, X_app), axis=1)
    else:
        ordre = np.argsort(-similarites_cosinus(X_nouveaux, X_app), axis=1)
    return voter(ordre[:, :k], y_app)

ks = [1, 3, 5, 7, 11, 15, 25]
resultats = {m: [np.mean(predire_voisins(X_val, X_app, y_app, k, m) == y_val) for k in ks] for m in ("distance", "cosinus")}

fig, ax = plt.subplots(figsize=(7.5, 3.6))
for m, couleur in [("distance", LIOUAI["navy"]), ("cosinus", LIOUAI["vert"])]:
    ax.plot(ks, resultats[m], "o-", color=couleur, label=m)
ax.set_xlabel("nombre de voisins k"); ax.set_ylabel("réussite (validation)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
for m in resultats:
    print(m, " ".join(f"k={k}: {r:.3f}" for k, r in zip(ks, resultats[m])))

**Activité** : Les deux mesures donnent-elles des résultats très différents ? Pourquoi, d'après vous, sur ces images-là ? Que se passe-t-il quand $k$ devient grand ?

Indiquez vos réponses et justifiez-les.

### Comprendre une erreur : regarder les voisins

In [ ]:
predits = predire_voisins(X_val, X_app, y_app, k=1)
print(f"k = 1 : {np.sum(predits != y_val)} erreurs sur {len(y_val)}")
k_err = np.flatnonzero(predits != y_val)[0]
ordre = np.argsort(distances_matricielles(X_val[k_err:k_err + 1], X_app), axis=1)[0, :9]
afficher(np.vstack([X_val[k_err], X_app[ordre]]),
         titres=[f"à lire ({y_val[k_err]})"] + [f"voisin : {y_app[o]}" for o in ordre], colonnes=10, taille=1.3)
plt.show()

### Le coût de la méthode

In [ ]:
debut = time.perf_counter()
predire_voisins(X_val, X_app, y_app, k=3)
t_voisins = (time.perf_counter() - debut) / len(X_val)
moyennes = np.array([X_app[y_app == c].mean(axis=0) for c in range(10)])
debut = time.perf_counter()
np.argmin(distances_matricielles(X_val, moyennes), axis=1)
t_moyenne = (time.perf_counter() - debut) / len(X_val)
print(f"Temps par image : voisins {1e6 * t_voisins:.0f} µs, chiffre moyen {1e6 * t_moyenne:.1f} µs")
print(f"Mémoire à conserver : voisins {X_app.size} nombres, chiffre moyen {moyennes.size} nombres")

## ⑤ Interpréter

In [ ]:
meilleur = max(((m, k, r) for m in resultats for k, r in zip(ks, resultats[m])), key=lambda t: t[2])
print(f"Vos paris : {pari_reussite_voisins:.0%} avec un voisin ; mesure préférée : {pari_meilleure_mesure}")
print(f"Résultats : {resultats['distance'][0]:.1%} avec un voisin (distance) ; meilleur réglage : {meilleur[0]}, k = {meilleur[1]} ({meilleur[2]:.1%})")

**Activité** :

1. Comparez les mérites des deux outils, chiffre moyen et plus proches voisins, sur trois critères : réussite, temps de calcul, mémoire. Lequel choisiriez-vous pour un appareil de tri qui traite 10 images par seconde ? Pour un téléphone avec peu de mémoire ?
2. Le gain de précision vous semble-t-il trop beau ? Qu'est-ce qui, dans la façon dont les données ont été collectées, pourrait l'expliquer ? (Indication : combien de personnes ont écrit ces chiffres ?)
3. Avez-vous une intuition de ce qui se passerait avec 100 fois plus d'images d'apprentissage ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Le concept s'applique-t-il ?** La distance traite les 64 pixels comme des coordonnées indépendantes et de même importance. Un chiffre décalé d'un pixel vers la droite est-il « proche » de l'original au sens de cette distance ? (L'étape C4 le mesurera.)
- **Un vote pondéré.** Un voisin très proche devrait peser plus qu'un voisin lointain.
- **Généraliser.** Écrivez une fonction `predire_pondere(X_nouveaux, X_app, y_app, k)` où chaque voisin vote avec le poids $1/(d + 1)$, $d$ étant sa distance. Comparez avec le vote simple pour $k = 5$ et $k = 15$.

In [ ]:
def predire_pondere(X_nouveaux, X_app, y_app, k=5):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def predire_pondere(X_nouveaux, X_app, y_app, k=5):
    D = distances_matricielles(X_nouveaux, X_app)
    proches = np.argsort(D, axis=1)[:, :k]
    poids = 1 / (np.take_along_axis(D, proches, axis=1) + 1)
    votes = np.zeros((len(X_nouveaux), 10))
    for c in range(10):
        votes[:, c] = (poids * (y_app[proches] == c)).sum(axis=1)
    return np.argmax(votes, axis=1)

for k in (5, 15):
    simple = np.mean(predire_voisins(X_val, X_app, y_app, k) == y_val)
    pondere = np.mean(predire_pondere(X_val, X_app, y_app, k) == y_val)
    print(f"k = {k:2d} : vote simple {simple:.3f}, vote pondéré {pondere:.3f}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Comparer les mérites des outils dans le contexte": 1,
    "Optimiser la rapidité et la réutilisabilité du calcul": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Corroborer ses résultats": 1,
    "Reconnaître si le concept s'applique": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "C2-plus-proches-voisins", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")